# ANIMA — Dataset Quality Check
Verify the preprocessing pipeline end-to-end:
```
source .mid  →  parse_mpe_midi()  →  MPETokenizer  →  export_53tet_mpe_midi()  →  audio
```
Each song type is A/B compared: **pipeline output** vs **original source**.

In [ ]:
import os, sys, json, random, importlib
from pathlib import Path
from IPython.display import Audio, display
import numpy as np
import mido

SRC_DIR = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
ROOT_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import tokenizer as tok_mod
import preprocess as preproc
import generate as gen

tokenizer = tok_mod.MPETokenizer()
print(f'Root: {ROOT_DIR}')
print(f'Vocab size: {tokenizer.vocab_size}')


In [ ]:
PLAYBACK_SPEED = 1.2
WAVEFORM       = 'sine'
REVERB         = 20
MIDI_TEMPO     = 120

QC_OUTPUT = ROOT_DIR / 'dataset' / 'generated' / 'qc'
QC_OUTPUT.mkdir(parents=True, exist_ok=True)
print('Config OK')


## Helper: export_53tet_mpe_midi

In [ ]:
def export_53tet_mpe_midi(tokens, output_path, tempo=120, velocity=80, pitch_offset=106):
    """
    Export a decoded token list to MPE MIDI with 53-TET pitch bends.
    tokens: list of token strings (e.g. decoded from vocab)
    """
    chords = gen.extract_chords_summary(tokens)
    if not chords:
        print('⚠️  No chords found in token sequence.')
        return None

    mid = mido.MidiFile(type=0, ticks_per_beat=480)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    track.append(mido.MetaMessage('set_tempo', tempo=int(60_000_000 / tempo)))
    tpb = mid.ticks_per_beat

    next_channel = 1
    for chord in chords:
        if not chord['pitches'] or chord['duration'] is None:
            continue
        dur_ticks = int(chord['duration'] * tpb)
        notes_in_chord = []
        for p53 in chord['pitches']:
            midi_note = max(0, min(127, round(p53 * 12.0 / 53.0)))
            residual = p53 - midi_note * 53.0 / 12.0
            bend = max(-8192, min(8191, int(round(residual * 1200.0 / 53.0 / 200.0 * 8192))))
            ch = next_channel
            next_channel = (next_channel % 15) + 1
            notes_in_chord.append((midi_note, bend, ch))
        for i, (note, bend, ch) in enumerate(notes_in_chord):
            track.append(mido.Message('pitchwheel', pitch=bend, channel=ch, time=0))
            track.append(mido.Message('note_on', note=note, velocity=velocity, channel=ch, time=0))
        for i, (note, bend, ch) in enumerate(notes_in_chord):
            track.append(mido.Message('note_off', note=note, velocity=0, channel=ch,
                                      time=dur_ticks if i == 0 else 0))
            track.append(mido.Message('pitchwheel', pitch=0, channel=ch, time=0))

    os.makedirs(os.path.dirname(str(output_path)) or '.', exist_ok=True)
    mid.save(str(output_path))
    n_notes = sum(len(c['pitches']) for c in chords if c['pitches'])
    n_bars  = sum(1 for t in tokens if t == 'BAR')
    print(f'Exported: {Path(output_path).name}  ({len(chords)} chords, {n_notes} notes, {n_bars} bars)')
    return str(output_path)


## 1. Discover Dataset Types

In [ ]:
import random, importlib, json, sys
from pathlib import Path
from IPython.display import Audio, display, HTML

# Reload modules to get the current version
sys.path.insert(0, str(SRC_DIR))
import tokenizer as tok_mod
import preprocess as preproc
importlib.reload(tok_mod)
importlib.reload(preproc)

tokenizer = tok_mod.MPETokenizer()

# Pick a few diverse source files (one per type)
MIDI_ROOT = ROOT_DIR / 'dataset' / 'midi_files' / '53_tet_mpe'
type_dirs = sorted(d for d in MIDI_ROOT.iterdir() if d.is_dir())
print(f"Found {len(type_dirs)} type directories:")
for d in type_dirs:
    n = len(list(d.glob('*.mid')))
    print(f"  {d.name}: {n:,} files")

## 2. Run Preprocessing Pipeline

In [ ]:
# Pick one random song from each type and run the full pipeline
QC_OUTPUT = ROOT_DIR / 'dataset' / 'generated' / 'qc'
QC_OUTPUT.mkdir(parents=True, exist_ok=True)

random.seed(42)
qc_results = []

for type_dir in type_dirs[:4]:   # first 4 types — enough to judge quality
    midi_files = list(type_dir.glob('*.mid'))
    if not midi_files:
        continue
    src_midi = random.choice(midi_files)

    # ── 1. Fresh preprocess ──
    result = preproc.preprocess_song(str(src_midi), tokenizer=tokenizer)
    if result is None:
        print(f"  SKIP (no chords): {src_midi.name}")
        continue

    # ── 2. Decode tokens for inspection ──
    vocab_data = json.load(open(ROOT_DIR / 'dataset' / 'tokenized' / 'vocab.json'))
    id2tok = {v: k for k, v in vocab_data['token_to_id'].items()}
    decoded = [id2tok.get(i, '?') for i in result['token_ids']]

    pv_steps = sorted(set(int(t.split('_')[1]) for t in decoded if t.startswith('PV_')))
    roots    = sorted(set(int(t.split('_')[1]) for t in decoded if t.startswith('ROOT_')))
    durs     = sorted(set(t for t in decoded if t.startswith('DUR_')))
    n_chords = result['n_chords']
    n_tokens = result['n_tokens']

    print(f"\n{'─'*60}")
    print(f"File:    {src_midi.name}")
    print(f"Type:    {type_dir.name}")
    print(f"Chords:  {n_chords}  |  Tokens: {n_tokens}")
    print(f"PV step range: {min(pv_steps)}–{max(pv_steps)}")
    midi_lo = round(min(pv_steps) * 12 / 53)
    midi_hi = round(max(pv_steps) * 12 / 53)
    note_names = ['C','C#','D','Eb','E','F','F#','G','Ab','A','Bb','B']
    print(f"MIDI range:    {midi_lo} ({note_names[midi_lo%12]}{midi_lo//12-1}) – "
          f"{midi_hi} ({note_names[midi_hi%12]}{midi_hi//12-1})")
    print(f"Roots (53-TET pc): {roots}")
    print(f"Durations: {durs}")
    print(f"First chord: {[t for t in decoded[:20] if not t.startswith('ROOT_')]}")

    # ── 3. Export to MPE MIDI ──
    out_path = QC_OUTPUT / f"qc_{type_dir.name}.mid"
    export_53tet_mpe_midi(decoded, out_path, tempo=MIDI_TEMPO,
                          pitch_offset=vocab_data.get('config', {}).get('pitch_offset', 106))

    qc_results.append({
        'name': src_midi.name,
        'type': type_dir.name,
        'src_midi': src_midi,
        'out_midi': out_path,
        'n_chords': n_chords,
        'pv_range': (min(pv_steps), max(pv_steps)),
    })

print(f"\n{'='*60}")
print(f"QC exports written to: {QC_OUTPUT}")


### A/B Listen: Fresh Pipeline vs Original Source

For each song:
- **Pipeline output** — source MIDI → preprocess → export → render (tests our full pipeline)
- **Original source** — plays the source `.mid` directly (ground truth)

If they sound the same, the pipeline is correct.

In [ ]:
import play_mpe as pm
importlib.reload(pm)

for r in qc_results:
    print(f"\n{'='*60}")
    print(f"Song: {r['name']}")
    print(f"Type: {r['type']}  |  Chords: {r['n_chords']}  |  PV steps: {r['pv_range'][0]}–{r['pv_range'][1]}")

    # ── Pipeline output ──
    print("\n▶ Pipeline output (preprocess → export → render):")
    audio, sr = pm.render_mpe_to_audio_data(
        str(r['out_midi']), speed=PLAYBACK_SPEED, waveform=WAVEFORM, reverb=REVERB)
    if audio is not None:
        display(Audio(audio, rate=sr))
    else:
        print("  ⚠️ No audio rendered")

    # ── Original source ──
    print("▶ Original source MIDI:")
    audio_src, sr_src = pm.render_mpe_to_audio_data(
        str(r['src_midi']), speed=PLAYBACK_SPEED, waveform=WAVEFORM, reverb=REVERB)
    if audio_src is not None:
        display(Audio(audio_src, rate=sr_src))
    else:
        print("  ⚠️ No audio from source MIDI")


### Pipeline integrity check
Verify the token → MIDI round-trip: decode the freshly preprocessed tokens back to chords
and check they match the source MIDI's note content.

In [ ]:
for r in qc_results[:2]:   # just first 2 for brevity
    src_midi = r['src_midi']
    print(f"\n{'─'*60}")
    print(f"Round-trip check: {r['name']}")

    # Source ground truth
    src_chords = tok_mod.parse_mpe_midi(str(src_midi))
    src_steps = [sorted(n['step_53'] for n in c['notes']) for c in src_chords[:5]]

    # Pipeline output
    result = preproc.preprocess_song(str(src_midi), tokenizer=tokenizer)
    vocab_data = json.load(open(ROOT_DIR / 'dataset' / 'tokenized' / 'vocab.json'))
    id2tok = {v: k for k, v in vocab_data['token_to_id'].items()}
    decoded = [id2tok.get(i,'?') for i in result['token_ids']]
    pipe_chords = gen.extract_chords_summary(decoded)
    pipe_steps = [sorted(c['pitches']) for c in pipe_chords[:5]]

    print(f"  {'Chord':<6}  {'Source steps':<35}  Pipeline steps")
    for i, (s, p) in enumerate(zip(src_steps, pipe_steps)):
        match = '✓' if s == p else '✗'
        print(f"  {i+1:<6}  {str(s):<35}  {p}  {match}")


## 4. Export Pipeline Outputs as WAV

In [ ]:
# --- Export QC pipeline outputs as WAV ---
import importlib
import play_mpe as pm
importlib.reload(pm)
from scipy.io import wavfile

audio_dir = ROOT_DIR / "dataset" / "audio"
audio_dir.mkdir(parents=True, exist_ok=True)

print(f"Exporting {len(qc_results)} QC file(s) → {audio_dir}")
for r in qc_results:
    wav_path = audio_dir / (r["out_midi"].stem + ".wav")
    if wav_path.exists():
        print(f"  skip (exists): {wav_path.name}")
        continue
    audio_data, sr = pm.render_mpe_to_audio_data(
        str(r["out_midi"]), speed=PLAYBACK_SPEED,
        waveform=WAVEFORM, reverb=REVERB,
    )
    if audio_data is not None:
        wavfile.write(str(wav_path), sr, audio_data.T)
        print(f"  saved: {wav_path.name}  ({r['type']})")
    else:
        print(f"  WARNING: no audio for {r['out_midi'].name}")

print("Done.")
